In [27]:
# ngrpr: 前年同期比の利益成長率 (%)
# 前年同期利益 > 0 かつ当期利益 > 0 の場合のみ算出。
# 赤字転落・黒字転換・赤字継続は成長率の意味が
# 不明瞭になるため null とする。

In [1]:
import polars as pl
from wequant.data_loading import read_data

In [2]:
df = read_data("../../data/datasets/prepared/quarterly-valuation-20260928_151110.parquet")

In [31]:
df.columns

['code', 'setd', 'annd', 'qtr', 'grsl', 'ngrpr', 'target']

In [32]:
df_grsl_decile = df.with_columns(
    pl.col("grsl")
    .qcut(
        10,
        labels=[str(i) for i in range(1, 11)],
    )
    .alias("grsl_decile")
)

In [33]:
(
    df_grsl_decile
    .filter(pl.col("grsl_decile").is_not_null())
    .group_by("grsl_decile")
    .agg(
        pl.len().alias("n"),
        pl.col("grsl").min().alias("grsl_min"),
        pl.col("grsl").max().alias("grsl_max"),
    )
    .sort(pl.col("grsl_decile").cast(pl.Int32))
)

grsl_decile,n,grsl_min,grsl_max
cat,u32,f64,f64
"""10""",2142,33.362243,16475.0
"""9""",2142,19.350811,33.347884
"""5""",2142,1.863586,4.93311
"""8""",2142,12.6563,19.345762
"""3""",2142,-7.400659,-1.918159
"""4""",2142,-1.917044,1.863102
"""6""",2142,4.933851,8.353412
"""7""",2142,8.35407,12.655253
"""2""",2142,-17.624442,-7.401405


In [34]:
result_grsl = (
    df_grsl_decile
    .filter(pl.col("grsl_decile").is_not_null())
    .group_by("grsl_decile")
    .agg(
        pl.len().alias("n"),
        pl.col("grsl").min().alias("grsl_min"),
        pl.col("grsl").max().alias("grsl_max"),
        pl.col("target").mean().alias("mean"),
        pl.col("target").median().alias("median"),
        (pl.col("target") > 0).mean().alias("win_rate"),
        pl.col("target").quantile(0.25).alias("p25"),
        pl.col("target").quantile(0.75).alias("p75"),
        pl.col("target").quantile(0.90).alias("p90"),
    )
    .sort(pl.col("grsl_decile").cast(pl.Int32))
)

result_grsl

grsl_decile,n,grsl_min,grsl_max,mean,median,win_rate,p25,p75,p90
cat,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""10""",2142,33.362243,16475.0,-0.444207,-0.646796,0.434641,-4.054011,2.459931,8.311195
"""9""",2142,19.350811,33.347884,-0.468668,-0.632418,0.421569,-3.595411,2.135343,7.429685
"""5""",2142,1.863586,4.93311,0.018523,-0.35747,0.457516,-2.390927,1.953691,5.535275
"""8""",2142,12.6563,19.345762,-0.4513,-0.591126,0.438842,-3.493209,2.135878,6.297962
"""3""",2142,-7.400659,-1.918159,-0.214385,-0.488673,0.426237,-2.493112,1.424263,4.847594
"""4""",2142,-1.917044,1.863102,0.035448,-0.273349,0.451914,-2.319323,1.623255,5.291278
"""6""",2142,4.933851,8.353412,-0.185198,-0.293186,0.454248,-2.765222,1.822183,5.388732
"""7""",2142,8.35407,12.655253,0.053769,-0.334602,0.457516,-2.829729,2.35914,6.574405
"""2""",2142,-17.624442,-7.401405,-0.517733,-0.688245,0.397292,-2.700119,1.279264,4.241913


In [35]:
result_qtr = (
    df
    .group_by("qtr")
    .agg(
        pl.len().alias("n"),
        pl.col("target").mean().alias("mean"),
        pl.col("target").median().alias("median"),
        (pl.col("target") > 0).mean().alias("win_rate"),
        pl.col("target").quantile(0.25).alias("p25"),
        pl.col("target").quantile(0.75).alias("p75"),
        pl.col("target").quantile(0.90).alias("p90"),
    )
    .sort("qtr")
)

result_qtr

qtr,n,mean,median,win_rate,p25,p75,p90
i64,u32,f64,f64,f64,f64,f64,f64
1,22044,-0.318422,-0.530253,0.432499,-2.944442,1.825796,5.844481


In [3]:
with pl.Config(tbl_rows=-1):
    print(
        df["target"].describe(
            percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]
        )
    )

shape: (15, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 22044.0    │
│ null_count ┆ 0.0        │
│ mean       ┆ -0.318422  │
│ std        ┆ 6.025471   │
│ min        ┆ -44.282172 │
│ 1%         ┆ -16.835224 │
│ 5%         ┆ -9.152491  │
│ 10%        ┆ -6.451485  │
│ 25%        ┆ -2.944442  │
│ 50%        ┆ -0.530178  │
│ 75%        ┆ 1.825796   │
│ 90%        ┆ 5.844481   │
│ 95%        ┆ 9.69106    │
│ 99%        ┆ 19.697488  │
│ max        ┆ 47.616589  │
└────────────┴────────────┘


In [28]:
df_decile = df.with_columns(
    pl.col("ngrpr")
    .qcut(
        10,
        labels=[str(i) for i in range(1, 11)],
    )
    .alias("ngrpr_decile")
)

In [30]:
result = (
    df_decile
    .filter(pl.col("ngrpr_decile").is_not_null())
    .group_by("ngrpr_decile")
    .agg(
        pl.len().alias("n"),
        pl.col("ngrpr").min().alias("ngrpr_min"),
        pl.col("ngrpr").max().alias("ngrpr_max"),
        pl.col("target").mean().alias("mean"),
        pl.col("target").median().alias("median"),
        (pl.col("target") > 0).mean().alias("win_rate"),
        pl.col("target").quantile(0.25).alias("p25"),
        pl.col("target").quantile(0.75).alias("p75"),
        pl.col("target").quantile(0.90).alias("p90"),
    )
    .sort(
        pl.col("ngrpr_decile").cast(pl.Int32)
    )
)

result

ngrpr_decile,n,ngrpr_min,ngrpr_max,mean,median,win_rate,p25,p75,p90
cat,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""10""",1471,111.541101,56230.286195,-0.428363,-0.539962,0.452073,-4.451131,2.904035,8.061058
"""9""",1471,57.107305,111.457165,-0.318546,-0.329386,0.459551,-3.525111,2.368104,6.863546
"""5""",1471,-4.329524,6.618323,0.047108,-0.174177,0.476547,-2.407408,2.152977,5.644072
"""8""",1471,33.383919,57.098368,-0.117068,-0.301067,0.468389,-3.211363,2.555944,7.242493
"""3""",1471,-35.342853,-17.111133,-0.257854,-0.434167,0.43032,-2.514284,1.602553,4.885409
"""4""",1471,-17.096451,-4.336672,-0.247076,-0.462597,0.433719,-2.666057,1.792021,5.611357
"""6""",1470,6.626147,18.134959,0.013575,-0.263812,0.463946,-2.714409,2.176307,6.127718
"""7""",1471,18.141621,33.353383,-0.318719,-0.638979,0.435758,-3.409878,2.154056,6.013112
"""2""",1471,-68.416985,-35.347359,-0.464722,-0.623785,0.414684,-2.638975,1.205039,4.31417


In [4]:
(
    df.with_columns(
        pl.col("setd").dt.year().alias("year")
    )
    .group_by("year")
    .agg(
        pl.len().alias("n"),
        pl.col("target").mean().alias("mean"),
        pl.col("target").median().alias("median"),
        (pl.col("target") > 0).mean().alias("win_rate"),
        pl.col("target").quantile(0.75).alias("p75"),
        pl.col("target").quantile(0.90).alias("p90"),
    )
    .sort("year")
)


year,n,mean,median,win_rate,p75,p90
i32,u32,f64,f64,f64,f64,f64
2020,3545,-0.587109,-0.832413,0.398307,1.564638,5.254701
2021,3649,-0.498688,-0.498962,0.42313,1.487784,4.925474
2022,3673,-0.470968,-0.630321,0.415464,1.451862,4.6943
2023,3752,-0.580504,-0.678547,0.409382,1.75284,6.071738
2024,3696,0.335382,0.004975,0.500271,2.540374,6.962532
2025,3729,-0.120658,-0.386003,0.447037,2.251353,6.641434


In [5]:
pdf = df
pdf = pdf.filter(pl.col("grsl")>=10).filter(pl.col("ngrpr")>=20)
 

In [6]:
(
    pdf.with_columns(
        pl.col("setd").dt.year().alias("year")
    )
    .group_by("year")
    .agg(
        pl.len().alias("n"),
        pl.col("target").mean().alias("mean"),
        pl.col("target").median().alias("median"),
        (pl.col("target") > 0).mean().alias("win_rate"),
        pl.col("target").quantile(0.75).alias("p75"),
        pl.col("target").quantile(0.90).alias("p90"),
    )
    .sort("year")
)


year,n,mean,median,win_rate,p75,p90
i32,u32,f64,f64,f64,f64,f64
2020,290,-1.474409,-1.727008,0.393103,2.235336,8.059307
2021,743,-0.569254,-0.435977,0.43607,1.874068,6.033096
2022,639,-0.680122,-0.646984,0.435055,2.39103,6.318399
2023,602,-0.88873,-0.888534,0.433555,2.724957,7.496654
2024,571,0.301936,-0.035871,0.495622,3.447974,9.25843
2025,527,-0.461545,-0.744307,0.434535,3.437045,8.821125


In [8]:
df_decile = df.with_columns(
    pl.col("ngrpr")
    .qcut(
        10,
        labels=[str(i) for i in range(1, 11)],
    )
    .alias("ngrpr_decile")
)
df_decile.group_by("ngrpr_decile").agg(
    pl.len().alias("n"),
    pl.col("ngrpr").min().alias("ngrpr_min"),
    pl.col("ngrpr").max().alias("ngrpr_max"),
).sort("ngrpr_decile")

ngrpr_decile,n,ngrpr_min,ngrpr_max
cat,u32,f64,f64
null,7335,null,null
"""1""",1471,-17072.981129,-68.423151
"""10""",1471,111.541101,56230.286195
"""2""",1471,-68.416985,-35.347359
"""3""",1471,-35.342853,-17.111133
…,…,…,…
"""5""",1471,-4.329524,6.618323
"""6""",1470,6.626147,18.134959
"""7""",1471,18.141621,33.353383


In [10]:
df.filter(
    pl.col("ngrpr").is_null()
).select([
    "code",
    "qtr",
    "prft",
    # 前年同期利益の列
]).head(30)

ColumnNotFoundError: unable to find column "prft"; valid columns: ["code", "setd", "annd", "qtr", "grsl", "ngrpr", "target"]

In [11]:
df.columns

['code', 'setd', 'annd', 'qtr', 'grsl', 'ngrpr', 'target']

In [12]:
ds_df = read_data("../../data/datasets/ds-quarterly-valuation-operating-announcement-20260928_150853.parquet")

In [13]:
ds_df

code,name,setd,annd,qtr,sls,prft,pr,grsl,dgrp,ngrpr,PER,divr,perf,bm
i64,str,date,date,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1301,"""極洋""",2018-09-30,2018-11-05,2,61245.0,507.0,0.827823,null,null,null,null,null,0.53267,-0.331529
1301,"""極洋""",2018-12-31,2019-02-08,3,78581.0,2208.0,2.80984,null,null,null,null,null,0.817844,-1.463576
1301,"""極洋""",2019-03-31,2019-05-13,4,58368.0,551.0,0.94401,null,null,null,null,null,0.0,-1.419075
1301,"""極洋""",2019-06-30,2019-08-02,1,62644.0,770.0,1.229168,null,null,null,null,null,0.475059,0.973225
1301,"""極洋""",2019-09-30,2019-11-05,2,64240.0,-35.0,-0.054483,4.890195,null,null,null,null,-0.069276,-1.117202
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
9997,"""ベルーナ""",2025-03-31,2025-05-13,4,52209.0,4546.0,8.707311,0.363322,2.560847,48.669421,11.462585,3.054259,-3.17623,0.827661
9997,"""ベルーナ""",2025-06-30,2025-07-31,1,50826.0,1591.0,3.130288,-1.055132,-1.123616,20.962201,9.727477,3.1205625,-2.909796,1.287047
9997,"""ベルーナ""",2025-09-30,2025-10-31,2,47166.0,3283.0,6.960522,3.074805,0.493959,24.778104,10.3,2.95,-0.725389,0.263312


In [26]:
from datetime import date
with pl.Config(tbl_rows=-1):
    print(
        ds_df.filter(
            (pl.col("code")==1301)
        ).select([
            "code",
            "name",
            "setd",
            "qtr",
            "sls",
            "prft",
            "pr",
            "grsl",
            "dgrp",
            "ngrpr",
        ]).tail(30)
    )


shape: (30, 10)
┌──────┬──────┬────────────┬─────┬───┬───────────┬────────────┬───────────┬─────────────┐
│ code ┆ name ┆ setd       ┆ qtr ┆ … ┆ pr        ┆ grsl       ┆ dgrp      ┆ ngrpr       │
│ ---  ┆ ---  ┆ ---        ┆ --- ┆   ┆ ---       ┆ ---        ┆ ---       ┆ ---         │
│ i64  ┆ str  ┆ date       ┆ i64 ┆   ┆ f64       ┆ f64        ┆ f64       ┆ f64         │
╞══════╪══════╪════════════╪═════╪═══╪═══════════╪════════════╪═══════════╪═════════════╡
│ 1301 ┆ 極洋 ┆ 2018-12-31 ┆ 3   ┆ … ┆ 2.80984   ┆ null       ┆ null      ┆ null        │
│ 1301 ┆ 極洋 ┆ 2019-03-31 ┆ 4   ┆ … ┆ 0.94401   ┆ null       ┆ null      ┆ null        │
│ 1301 ┆ 極洋 ┆ 2019-06-30 ┆ 1   ┆ … ┆ 1.229168  ┆ null       ┆ null      ┆ null        │
│ 1301 ┆ 極洋 ┆ 2019-09-30 ┆ 2   ┆ … ┆ -0.054483 ┆ 4.890195   ┆ null      ┆ null        │
│ 1301 ┆ 極洋 ┆ 2019-12-31 ┆ 3   ┆ … ┆ 2.394592  ┆ -0.036905  ┆ 11.275862 ┆ -44.081201  │
│ 1301 ┆ 極洋 ┆ 2020-03-31 ┆ 4   ┆ … ┆ 0.529054  ┆ -2.201549  ┆ 0.193774  ┆ -34.706697  │
│ 1301

In [15]:
df.filter(
    pl.col("ngrpr").is_not_null() &
    (pl.col("grsl") < 0)
).select([
    "code",
    "setd",
    "qtr",
    "grsl",
    "ngrpr",
]).head(20)

code,setd,qtr,grsl,ngrpr
i64,date,i64,f64,f64
1301,2020-06-30,1,-13.287785,401.401115
1301,2023-06-30,1,-4.465243,-0.391259
1332,2020-06-30,1,-12.742976,-26.997439
1333,2020-06-30,1,-14.274898,-12.697919
1352,2020-06-30,1,-1.664437,19.188554
…,…,…,…,…
1407,2020-11-30,1,-6.902998,34.80367
1407,2022-11-30,1,-56.586105,-8.036544
1407,2024-11-30,1,-40.622533,-84.645001


In [17]:
df.filter(
    pl.col("dgrp").is_not_null() &
    (pl.col("dgrp") < 0)
).select([
    "code",
    "setd",
    "qtr",
    "grsl",
    "ngrpr",
]).head(20)

ColumnNotFoundError: unable to find column "dgrp"; valid columns: ["code", "setd", "annd", "qtr", "grsl", "ngrpr", "target"]

In [ ]:
ds_df.filter(
    (pl.col("setd").dt.year() >= 2020) &
    pl.col("ngrpr").is_null()
).select([
    "code",
    "name",
    "setd",
    "qtr",
    "sls",
    "prft",
    "pr",
    "grsl",
    "dgrp",
    "ngrpr",
]).head(30)